# BetterCallNLI — MS3 Evaluation on Kaggle

Runs the full multi-agent pipeline (Router → Analyst → Reviewer) on the ContractNLI test split and produces the MS3 deliverables.

**No GPU needed.** Per spec §2f the fine-tuned model is NOT used. All LLM calls go to HuggingFace Serverless (`Qwen/Qwen2.5-7B-Instruct`); retrieval uses ChromaDB Cloud or Neo4j Aura. Kaggle is just remote CPU compute.

## Prereqs

1. **Upload the repo as a Kaggle Dataset** (zip `BetterCallNLI/` and add to the notebook's inputs). Update `REPO_DIR` in Cell 2 to the mount path.
2. **Add secrets** under “Add-ons → Secrets”:
   - `HF_TOKEN` (required)
   - `CHROMA_API_KEY` (required when `RETRIEVAL_MODE = 'vector'`)
   - `NEO4J_URI`, `NEO4J_USERNAME`, `NEO4J_PASSWORD` (required when `RETRIEVAL_MODE = 'graphrag'`)
3. **Accelerator:** None or CPU. (You can use a GPU session if available — the agents won't use it.)

## What this notebook produces (matches the local `cli.py --mode evaluate` outputs)

- `predictions_ms3.json` — verdicts per contract (§3a, §3b)
- `runtraces/runtrace_<id>.json` — per-contract runtraces (§2c, §2h, §3d)
- `evaluation_metrics_ms3.csv` — MS3 metrics with the MS1 CSV header (§3e)
- `evaluation_metrics_combined.csv` — MS1 + MS3 row (§5b deliverable)
- `evaluation_metrics_ms3.json` — full breakdown
- `runtraces_ms3.zip` — zipped runtraces (§5c deliverable)

## Cell 1 · Install dependencies

In [ ]:
!pip install -q rich tqdm pandas pyyaml python-dotenv \
  huggingface_hub sentence-transformers chromadb neo4j kagglehub

## Cell 2 · Configure paths + secrets

In [ ]:
import os, sys
from pathlib import Path

# ---- UPDATE if your dataset mount path differs ------------------------------
REPO_DIR        = Path('/kaggle/input/bettercallnli/BetterCallNLI')
PLAYBOOK_PATH   = REPO_DIR / 'playbook.yaml'
MS1_CSV_PATH    = REPO_DIR / 'results' / 'evaluation_metrics.csv'  # for the combined CSV (§5b)

# ---- Optional tuning --------------------------------------------------------
RETRIEVAL_MODE  = 'vector'                       # 'vector' or 'graphrag'
LIMIT_CONTRACTS = None                           # set to int (e.g. 5) for a smoke run
OUTPUT_DIR      = Path('/kaggle/working/outputs/ms3')

# ---- Pull Kaggle Secrets into os.environ -----------------------------------
try:
    from kaggle_secrets import UserSecretsClient
    secrets = UserSecretsClient()
    for k in ('HF_TOKEN', 'CHROMA_API_KEY',
              'NEO4J_URI', 'NEO4J_USERNAME', 'NEO4J_PASSWORD'):
        try:
            os.environ[k] = secrets.get_secret(k)
        except Exception:
            pass
    have = [k for k in ('HF_TOKEN','CHROMA_API_KEY','NEO4J_URI') if os.getenv(k)]
    print('Loaded secrets:', have)
except ImportError:
    print('kaggle_secrets not available; expecting env vars already set')

# ---- Make the repo importable + verify paths --------------------------------
sys.path.insert(0, str(REPO_DIR))
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print('REPO_DIR     :', REPO_DIR, '(exists)' if REPO_DIR.exists() else '(MISSING)')
print('PLAYBOOK     :', PLAYBOOK_PATH, '(exists)' if PLAYBOOK_PATH.exists() else '(MISSING)')
print('MS1 CSV      :', MS1_CSV_PATH, '(exists)' if MS1_CSV_PATH.exists() else '(NOT FOUND — combined CSV will only have MS3 row)')
print('OUTPUT_DIR   :', OUTPUT_DIR)

## Cell 3 · Build orchestrator + smoke test on 1 contract

In [ ]:
from src.agent.orchestrator import build_orchestrator
from src.utils.contract_loader import get_test_contracts

orchestrator = build_orchestrator(
    retrieval_mode = RETRIEVAL_MODE,
    playbook_path  = str(PLAYBOOK_PATH),
)
print('Orchestrator built.')
print('  retrieval :', orchestrator.retriever.mode)
print('  llm model :', orchestrator.router.model)

contracts = get_test_contracts()  # kagglehub download
print(f'Loaded {len(contracts)} test contracts.')

# Smoke run — one contract, prints first few verdicts
import time
smoke = contracts[0]
t0 = time.perf_counter()
smoke_out = orchestrator.hypothesis_pipeline.run(smoke)
print(f'\nSmoke run on {smoke["id"]} took {time.perf_counter()-t0:.1f}s — {len(smoke_out["verdicts"])} verdicts')
for v in smoke_out['verdicts'][:3]:
    print(f'  {v["hypothesis_id"]:>4}  {v["label"]:<14}  conf={v["confidence"]:.2f}  ev={len(v.get("evidence",[]))}')

## Cell 4 · Run the full evaluation

In [ ]:
import json
from scripts.evaluate_ms3 import run_evaluation
from tqdm.auto import tqdm

total = LIMIT_CONTRACTS or len(contracts)
bar = tqdm(total=total, desc='evaluate')

def _progress(i, n, c_id, *, status='ok', latency_ms=0.0):
    bar.set_postfix_str(f'{c_id} [{status}] {latency_ms/1000:.1f}s')
    bar.update(1)

metrics = run_evaluation(
    orchestrator  = orchestrator,
    contracts     = contracts,
    output_dir    = OUTPUT_DIR,
    limit         = LIMIT_CONTRACTS,
    progress_cb   = _progress,
    playbook_path = PLAYBOOK_PATH,
    ms1_csv_path  = MS1_CSV_PATH if MS1_CSV_PATH.exists() else None,
)
bar.close()

print('\n=== Aggregate metrics ===')
print(json.dumps({k: v for k, v in metrics.items() if k != 'per_hypothesis'}, indent=2))

## Cell 5 · Per-hypothesis breakdown + final file list

In [ ]:
import pandas as pd

rows = [
    {'hypothesis': h, 'correct': v['correct'], 'total': v['total'], 'accuracy': v['accuracy']}
    for h, v in sorted(metrics['per_hypothesis'].items())
]
df = pd.DataFrame(rows).sort_values('hypothesis')
print(df.to_string(index=False))

print('\nOutput files:')
for p in sorted(OUTPUT_DIR.rglob('*')):
    if p.is_file():
        print(f'  {p.relative_to(OUTPUT_DIR)}  ({p.stat().st_size:,} bytes)')

# Combined CSV peek
combined = OUTPUT_DIR / 'evaluation_metrics_combined.csv'
if combined.exists():
    print('\nevaluation_metrics_combined.csv:')
    print(combined.read_text())

## Done

Download `OUTPUT_DIR / 'runtraces_ms3.zip'` for the §5c deliverable and `evaluation_metrics_combined.csv` for the §5b deliverable.

If Member 4's `src/enrichment/playbook_enricher.py` and `src/utils/runtrace.py` are merged into this branch, the evaluation runner picks them up automatically (see the soft-import block at the top of `scripts/evaluate_ms3.py`). No code change needed.